<a href="https://colab.research.google.com/github/Gabriela-Sol/VpC2---Deteccion-de-humo-y-fuego/blob/main/02_entrenamiento_YOLO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 - Entrenamiento de modelos YOLO para detección de humo y fuego

- Prueba iniciaal con yolov8n
## Salidas esperadas

El entrenamiento genera:

- pesos del modelo (`best.pt` y `last.pt`);
- métricas de entrenamiento y validación;
- curvas de desempeño;
- matriz de confusión;
- carpeta de resultados asociada al experimento.

In [2]:
# ============================================================
# Setup general del entorno
# ============================================================

from pathlib import Path
import os
import sys
import random
import shutil
import time
import yaml

SEED = 42
random.seed(SEED)

IN_COLAB = "google.colab" in sys.modules

print("Ejecutando en Google Colab:", IN_COLAB)
print("Directorio actual:", Path.cwd())

Ejecutando en Google Colab: True
Directorio actual: /content


In [3]:
# ============================================================
# Instalación de dependencias
# ============================================================

REPO_URL = "https://github.com/Gabriela-Sol/VpC2---Deteccion-de-humo-y-fuego"
REPO_NAME = "VpC2---Deteccion-de-humo-y-fuego"

if IN_COLAB:
    !pip install -q -r https://raw.githubusercontent.com/Gabriela-Sol/VpC2---Deteccion-de-humo-y-fuego/main/requirements.txt

print("Dependencias instaladas.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.7 MB/s eta 0:00:00
Dependencias instaladas.


In [4]:
# ============================================================
# Verificación de GPU
# ============================================================

import torch

print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No se detectó GPU. Para entrenar se recomienda activar GPU en Colab.")

CUDA disponible: True
GPU: Tesla T4


In [5]:
# ============================================================
# Montar Google Drive
# ============================================================

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/VCII_DFire")
DRIVE_RUNS_DIR = DRIVE_PROJECT_DIR / "runs"
DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("Carpeta principal en Drive:", DRIVE_PROJECT_DIR)
print("Carpeta de corridas:", DRIVE_RUNS_DIR)

Mounted at /content/drive
Carpeta principal en Drive: /content/drive/MyDrive/VCII_DFire
Carpeta de corridas: /content/drive/MyDrive/VCII_DFire/runs


In [6]:
# ============================================================
# Clonado o actualización del repositorio
# ============================================================

PROJECT_DIR = Path("/content") / REPO_NAME

if IN_COLAB:
    if PROJECT_DIR.exists():
        print("El repositorio ya existe. Actualizando...")
        %cd {PROJECT_DIR}
        !git pull
    else:
        print("Clonando repositorio...")
        %cd /content
        !git clone {REPO_URL}.git
        %cd {PROJECT_DIR}
else:
    PROJECT_DIR = Path.cwd()

print("PROJECT_DIR:", PROJECT_DIR)
print("Contenido del proyecto:")
print(os.listdir(PROJECT_DIR))

Clonando repositorio...
/content
Cloning into 'VpC2---Deteccion-de-humo-y-fuego'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 77 (delta 32), reused 9 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (77/77), 2.27 MiB | 7.80 MiB/s, done.
Resolving deltas: 100% (32/32), done.
/content/VpC2---Deteccion-de-humo-y-fuego
PROJECT_DIR: /content/VpC2---Deteccion-de-humo-y-fuego
Contenido del proyecto:
['.gitignore', 'demo', 'README.md', 'data', '.git', 'src', 'notebooks', 'requirements.txt', '02_entrenamiento_YOLO.ipynb', 'configs', 'reports']


In [7]:
# ============================================================
# Carga de configuración del experimento
# ============================================================

EXPERIMENT_CONFIG_PATH = PROJECT_DIR / "configs" / "experiments" / "yolov8n_baseline.yaml"

if not EXPERIMENT_CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo de configuración: {EXPERIMENT_CONFIG_PATH}"
    )

with open(EXPERIMENT_CONFIG_PATH, "r", encoding="utf-8") as file:
    experiment_config = yaml.safe_load(file)

experiment_name = experiment_config["experiment"]["name"]
model_name = experiment_config["experiment"]["model"]
family = experiment_config["experiment"]["family"]

print("Experimento:", experiment_name)
print("Familia:", family)
print("Modelo:", model_name)
print("Descripción:", experiment_config["experiment"]["description"])
print("Config path:", EXPERIMENT_CONFIG_PATH)

Experimento: yolov8n_baseline
Familia: YOLOv8
Modelo: yolov8n.pt
Descripción: Baseline liviano con YOLOv8 nano.
Config path: /content/VpC2---Deteccion-de-humo-y-fuego/configs/experiments/yolov8n_baseline.yaml


In [8]:
# ============================================================
# Descarga o localización del dataset
# ============================================================

import kagglehub

DATASET_ID = "sayedgamal99/smoke-fire-detection-yolo"

dataset_root = Path(kagglehub.dataset_download(DATASET_ID))

print("Dataset descargado/localizado en:")
print(dataset_root)

print("\nContenido del directorio raíz del dataset:")
for item in dataset_root.iterdir():
    print("-", item)

Using Colab cache for faster access to the 'smoke-fire-detection-yolo' dataset.
Dataset descargado/localizado en:
/kaggle/input/smoke-fire-detection-yolo

Contenido del directorio raíz del dataset:
- /kaggle/input/smoke-fire-detection-yolo/data.yaml
- /kaggle/input/smoke-fire-detection-yolo/data


In [9]:
# ============================================================
# Detección estructura del dataset
# ============================================================

def find_yolo_dataset_dir(root: Path) -> Path:
    """
    Busca automáticamente la carpeta que contiene la estructura esperada:
    train/images, train/labels, val/images, val/labels.
    """
    candidates = [root] + [p for p in root.rglob("*") if p.is_dir()]

    for candidate in candidates:
        train_images = candidate / "train" / "images"
        train_labels = candidate / "train" / "labels"
        val_images = candidate / "val" / "images"
        val_labels = candidate / "val" / "labels"

        if (
            train_images.exists()
            and train_labels.exists()
            and val_images.exists()
            and val_labels.exists()
        ):
            return candidate

    raise FileNotFoundError(
        "No se encontró una estructura YOLO válida con train/images, train/labels, val/images y val/labels."
    )

DATA_DIR = find_yolo_dataset_dir(dataset_root)

print("Carpeta de datos YOLO detectada:")
print(DATA_DIR)

for split in ["train", "val", "test"]:
    split_dir = DATA_DIR / split
    print(f"{split}: existe={split_dir.exists()} -> {split_dir}")

Carpeta de datos YOLO detectada:
/kaggle/input/smoke-fire-detection-yolo/data
train: existe=True -> /kaggle/input/smoke-fire-detection-yolo/data/train
val: existe=True -> /kaggle/input/smoke-fire-detection-yolo/data/val
test: existe=True -> /kaggle/input/smoke-fire-detection-yolo/data/test


In [10]:
# ============================================================
# Creación del archivo YAML para YOLO
# ============================================================

DFIRE_YAML = Path("/content/dfire_colab.yaml")

dfire_config = {
    "path": str(DATA_DIR),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "nc": 2,
    "names": {
        0: "smoke",
        1: "fire",
    },
}

with open(DFIRE_YAML, "w", encoding="utf-8") as file:
    yaml.safe_dump(dfire_config, file, sort_keys=False, allow_unicode=True)

print("Archivo YAML creado en:")
print(DFIRE_YAML)

print("\nContenido:")
with open(DFIRE_YAML, "r", encoding="utf-8") as file:
    print(file.read())

Archivo YAML creado en:
/content/dfire_colab.yaml

Contenido:
path: /kaggle/input/smoke-fire-detection-yolo/data
train: train/images
val: val/images
test: test/images
nc: 2
names:
  0: smoke
  1: fire



In [11]:
# # ============================================================
# validación rápida del dataset
# ============================================================

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def count_split_files(data_dir: Path, split:str):
  images_dir = data_dir / split / "images"
  labels_dir = data_dir / split / "labels"

  image_files = [f for f in images_dir.iterdir() if f.suffix.lower() in IMAGE_EXTENSIONS]
  label_files = [f for f in labels_dir.iterdir() if f.suffix.lower() == ".txt"]

  return len(image_files), len(label_files)

for split in ["train", "val", "test"]:
  images_count, labels_count = count_split_files(DATA_DIR, split)
  print(f"{split:5s} | imágenes: {images_count:6d} | etiquetas: {labels_count:6d}")




train | imágenes:  14122 | etiquetas:  14122
val   | imágenes:   3099 | etiquetas:   3099
test  | imágenes:   4306 | etiquetas:   4306


In [12]:
# ============================================================
# Validación básica de etiquetas YOLO
# ============================================================

def validate_yolo_labels(labels_dir: Path):
    """
    Valida etiquetas YOLO:
    class_id x_center y_center width height
    con coordenadas normalizadas entre 0 y 1.
    """
    invalid_entries = []
    empty_files = 0
    total_boxes = 0

    for label_path in labels_dir.glob("*.txt"):
        content = label_path.read_text().strip()

        if content == "":
            empty_files += 1
            continue

        for line_number, line in enumerate(content.splitlines(), start=1):
            parts = line.split()

            if len(parts) != 5:
                invalid_entries.append((str(label_path), line_number, "Cantidad de campos != 5", line))
                continue

            try:
                class_id = int(float(parts[0]))
                coords = list(map(float, parts[1:]))
            except ValueError:
                invalid_entries.append((str(label_path), line_number, "Valores no numéricos", line))
                continue

            if class_id not in [0, 1]:
                invalid_entries.append((str(label_path), line_number, "Clase fuera de rango", line))
                continue

            if not all(0 <= value <= 1 for value in coords):
                invalid_entries.append((str(label_path), line_number, "Coordenadas fuera de [0, 1]", line))
                continue

            total_boxes += 1

    return {
        "empty_files": empty_files,
        "total_boxes": total_boxes,
        "invalid_entries": invalid_entries,
    }

for split in ["train", "val", "test"]:
    labels_dir = DATA_DIR / split / "labels"
    validation = validate_yolo_labels(labels_dir)

    print(f"\nSplit: {split}")
    print("Labels vacíos:", validation["empty_files"])
    print("Bounding boxes válidos:", validation["total_boxes"])
    print("Entradas problemáticas:", len(validation["invalid_entries"]))

    if validation["invalid_entries"][:5]:
        print("Primeras entradas problemáticas:")
        for entry in validation["invalid_entries"][:5]:
            print(entry)


Split: train
Labels vacíos: 6458
Bounding boxes válidos: 17432
Entradas problemáticas: 0

Split: val
Labels vacíos: 1375
Bounding boxes válidos: 3932
Entradas problemáticas: 0

Split: test
Labels vacíos: 2005
Bounding boxes válidos: 5185
Entradas problemáticas: 8
Primeras entradas problemáticas:
('/kaggle/input/smoke-fire-detection-yolo/data/test/labels/WEB11606.txt', 1, 'Coordenadas fuera de [0, 1]', '0 0.2578125 0.4986111111111111 0.50625 1.0027777777777778')
('/kaggle/input/smoke-fire-detection-yolo/data/test/labels/WEB11600.txt', 1, 'Coordenadas fuera de [0, 1]', '0 0.496875 0.4222222222222222 1.0562500000000001 0.8388888888888889')
('/kaggle/input/smoke-fire-detection-yolo/data/test/labels/WEB11090.txt', 1, 'Coordenadas fuera de [0, 1]', '0 0.5437500000000001 0.5013888888888889 0.5375 1.0027777777777778')
('/kaggle/input/smoke-fire-detection-yolo/data/test/labels/WEB10821.txt', 3, 'Coordenadas fuera de [0, 1]', '0 0.49843750000000003 0.3388888888888889 1.0093750000000001 0.616666

In [13]:
# ============================================================
# Función de entrenamiento desde configuración
# ============================================================

from ultralytics import YOLO
import pandas as pd

def train_from_config(config: dict, data_yaml: Path) -> dict:
    """
    Entrena un modelo YOLO a partir de un archivo de configuración YAML.
    Los resultados se guardan en una carpeta específica por experimento.
    """

    exp = config["experiment"]
    train_cfg = config["training"]
    output_cfg = config["output"]

    experiment_name = exp["name"]
    model_name = exp["model"]
    project_dir = Path(output_cfg["project"])

    project_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 80)
    print(f"Experimento: {experiment_name}")
    print(f"Familia: {exp['family']}")
    print(f"Modelo: {model_name}")
    print(f"Resultados en: {project_dir / experiment_name}")
    print("=" * 80)

    start_time = time.time()

    model = YOLO(model_name)

    results = model.train(
        data=str(data_yaml),
        epochs=train_cfg["epochs"],
        imgsz=train_cfg["imgsz"],
        batch=train_cfg["batch"],
        patience=train_cfg["patience"],
        optimizer=train_cfg["optimizer"],
        lr0=train_cfg["lr0"],
        seed=train_cfg["seed"],
        project=str(project_dir),
        name=experiment_name,
        exist_ok=True,
        plots=True,
    )

    elapsed_time = time.time() - start_time

    experiment_dir = project_dir / experiment_name
    best_model_path = experiment_dir / "weights" / "best.pt"
    last_model_path = experiment_dir / "weights" / "last.pt"
    results_csv = experiment_dir / "results.csv"

    summary = {
        "experiment": experiment_name,
        "family": exp["family"],
        "model": model_name,
        "epochs": train_cfg["epochs"],
        "imgsz": train_cfg["imgsz"],
        "batch": train_cfg["batch"],
        "training_time_min": round(elapsed_time / 60, 2),
        "experiment_dir": str(experiment_dir),
        "best_model_path": str(best_model_path),
        "last_model_path": str(last_model_path),
        "results_csv": str(results_csv),
        "best_exists": best_model_path.exists(),
        "last_exists": last_model_path.exists(),
    }

    print("\nEntrenamiento finalizado.")
    print("Tiempo total [min]:", summary["training_time_min"])
    print("Best model:", best_model_path)
    print("Last model:", last_model_path)

    return summary

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [14]:
# ============================================================
# Ejecución del experimento baseline
# ============================================================

RUN_TRAINING = True

if RUN_TRAINING:
    experiment_summary = train_from_config(
        config=experiment_config,
        data_yaml=DFIRE_YAML,
    )

    summary_df = pd.DataFrame([experiment_summary])
    display(summary_df)
else:
    print("RUN_TRAINING=False. No se ejecutó el entrenamiento.")

Experimento: yolov8n_baseline
Familia: YOLOv8
Modelo: yolov8n.pt
Resultados en: /content/drive/MyDrive/VCII_DFire/runs/yolov8n_baseline
Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dfire_colab.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mo

,experiment,family,model,epochs,imgsz,batch,training_time_min,experiment_dir,best_model_path,last_model_path,results_csv,best_exists,last_exists
0,yolov8n_baseline,YOLOv8,yolov8n.pt,30,640,16,147.54,/content/drive/MyDrive/VCII_DFire/runs/yolov8n...,/content/drive/MyDrive/VCII_DFire/runs/yolov8n...,/content/drive/MyDrive/VCII_DFire/runs/yolov8n...,/content/drive/MyDrive/VCII_DFire/runs/yolov8n...,True,True


In [15]:
# ============================================================
# Verificación de pesos guardados
# ============================================================

experiment_name = experiment_config["experiment"]["name"]
project_dir = Path(experiment_config["output"]["project"])
experiment_dir = project_dir / experiment_name

best_model_path = experiment_dir / "weights" / "best.pt"
last_model_path = experiment_dir / "weights" / "last.pt"

print("Carpeta del experimento:", experiment_dir)
print("Best model existe:", best_model_path.exists(), best_model_path)
print("Last model existe:", last_model_path.exists(), last_model_path)

Carpeta del experimento: /content/drive/MyDrive/VCII_DFire/runs/yolov8n_baseline
Best model existe: True /content/drive/MyDrive/VCII_DFire/runs/yolov8n_baseline/weights/best.pt
Last model existe: True /content/drive/MyDrive/VCII_DFire/runs/yolov8n_baseline/weights/last.pt


In [16]:
# ============================================================
# Guardar resumen general de experimentos
# ============================================================

SUMMARY_PATH = DRIVE_PROJECT_DIR / "experiment_summary.csv"

if RUN_TRAINING:
    if SUMMARY_PATH.exists():
        previous_summary = pd.read_csv(SUMMARY_PATH)
        updated_summary = pd.concat(
            [previous_summary, summary_df],
            ignore_index=True
        )
    else:
        updated_summary = summary_df.copy()

    updated_summary.to_csv(SUMMARY_PATH, index=False)
    display(updated_summary)

    print("Resumen actualizado en:", SUMMARY_PATH)
else:
    print("No se actualizó el resumen porque no se entrenó.")

,experiment,family,model,epochs,imgsz,batch,training_time_min,experiment_dir,best_model_path,last_model_path,results_csv,best_exists,last_exists
0,yolov8n_baseline,YOLOv8,yolov8n.pt,30,640,16,147.54,/content/drive/MyDrive/VCII_DFire/runs/yolov8n...,/content/drive/MyDrive/VCII_DFire/runs/yolov8n...,/content/drive/MyDrive/VCII_DFire/runs/yolov8n...,/content/drive/MyDrive/VCII_DFire/runs/yolov8n...,True,True


Resumen actualizado en: /content/drive/MyDrive/VCII_DFire/experiment_summary.csv


In [17]:
# ============================================================
# Copiar resultados principales al repositorio local
# ============================================================

REPORTS_RESULTS_DIR = PROJECT_DIR / "reports" / "results" / experiment_name
REPORTS_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

files_to_copy = [
    "results.csv",
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "F1_curve.png",
    "P_curve.png",
    "R_curve.png",
]

for filename in files_to_copy:
    src = experiment_dir / filename
    dst = REPORTS_RESULTS_DIR / filename

    if src.exists():
        shutil.copy(src, dst)
        print("Copiado:", dst)
    else:
        print("No encontrado:", src)

# Guardar también la configuración usada para este experimento
used_config_path = REPORTS_RESULTS_DIR / "experiment_config_used.yaml"

with open(used_config_path, "w", encoding="utf-8") as file:
    yaml.safe_dump(experiment_config, file, sort_keys=False, allow_unicode=True)

print("Config usada guardada en:", used_config_path)

Copiado: /content/VpC2---Deteccion-de-humo-y-fuego/reports/results/yolov8n_baseline/results.csv
Copiado: /content/VpC2---Deteccion-de-humo-y-fuego/reports/results/yolov8n_baseline/results.png
Copiado: /content/VpC2---Deteccion-de-humo-y-fuego/reports/results/yolov8n_baseline/confusion_matrix.png
Copiado: /content/VpC2---Deteccion-de-humo-y-fuego/reports/results/yolov8n_baseline/confusion_matrix_normalized.png
No encontrado: /content/drive/MyDrive/VCII_DFire/runs/yolov8n_baseline/PR_curve.png
No encontrado: /content/drive/MyDrive/VCII_DFire/runs/yolov8n_baseline/F1_curve.png
No encontrado: /content/drive/MyDrive/VCII_DFire/runs/yolov8n_baseline/P_curve.png
No encontrado: /content/drive/MyDrive/VCII_DFire/runs/yolov8n_baseline/R_curve.png
Config usada guardada en: /content/VpC2---Deteccion-de-humo-y-fuego/reports/results/yolov8n_baseline/experiment_config_used.yaml


In [18]:
# ============================================================
# Commit y push seguro de resultados al repositorio real de GitHub
# ============================================================

from getpass import getpass
import subprocess
from pathlib import Path

%cd {PROJECT_DIR}

# ------------------------------------------------------------
# Configuración de usuario para el commit
# ------------------------------------------------------------

!git config user.name "Gabriela-Sol"
!git config user.email "solgab.salazar@gmail.com"

print("Estado inicial:")
!git status -sb

# ------------------------------------------------------------
# Traer cambios remotos antes de pushear
# ------------------------------------------------------------

print("\nActualizando rama local con cambios remotos...")
pull_result = subprocess.run(
    ["git", "pull", "--rebase", "origin", "main"],
    text=True,
    capture_output=True,
)

print(pull_result.stdout)
print(pull_result.stderr)

if pull_result.returncode != 0:
    print("Hubo un problema durante el pull --rebase.")
    print("Revisar si hay conflictos antes de continuar.")
    raise RuntimeError("No se pudo completar git pull --rebase.")

print("\nEstado después del pull --rebase:")
!git status -sb

# ------------------------------------------------------------
# Agregar archivos del experimento
# ------------------------------------------------------------

paths_to_add = [
    f"reports/results/{experiment_name}/",
    "configs/experiments/yolov8n_baseline.yaml",
]

print("\nAgregando archivos al commit:")

for path in paths_to_add:
    if Path(path).exists():
        subprocess.run(["git", "add", path], check=True)
        print("Agregado:", path)
    else:
        print("No existe, se omite:", path)

print("\nEstado después de git add:")
status_result = subprocess.run(
    ["git", "status", "--short"],
    text=True,
    capture_output=True,
)

print(status_result.stdout)

# ------------------------------------------------------------
# Commit solo si hay cambios
# ------------------------------------------------------------

if not status_result.stdout.strip():
    print("No hay cambios nuevos para commitear.")
else:
    commit_message = f"results: update {experiment_name} outputs"

    commit_result = subprocess.run(
        ["git", "commit", "-m", commit_message],
        text=True,
        capture_output=True,
    )

    print("\nSalida del commit:")
    print(commit_result.stdout)
    print(commit_result.stderr)

    if commit_result.returncode != 0:
        print("No se pudo crear el commit.")
        raise RuntimeError("Error en git commit.")

# ------------------------------------------------------------
# Push seguro con token
# ------------------------------------------------------------

print("\nEstado antes del push:")
!git status -sb

github_token = getpass("Pegá tu GitHub token: ")

repo_url = (
    f"https://x-access-token:{github_token}"
    "@github.com/Gabriela-Sol/VpC2---Deteccion-de-humo-y-fuego.git"
)

push_result = subprocess.run(
    ["git", "push", repo_url, "HEAD:main"],
    text=True,
    capture_output=True,
)

def sanitize(text):
    if github_token:
        return text.replace(github_token, "***TOKEN***")
    return text

print("\nReturn code push:", push_result.returncode)

if push_result.returncode == 0:
    print("Push realizado correctamente.")
else:
    print("Error en el push:")
    print(sanitize(push_result.stdout))
    print(sanitize(push_result.stderr))

github_token = None
repo_url = None

print("\nEstado final:")
!git status -sb

/content/VpC2---Deteccion-de-humo-y-fuego
Estado inicial:
## main...origin/main
?? reports/results/

Actualizando rama local con cambios remotos...
Updating 3b586ba..fb51eb3
Fast-forward
 02_entrenamiento_YOLO.ipynb                      | 1098 +++++-
 notebooks/01_exploracion_dfire.ipynb             | 3878 ++++++++++++++++++----
 reports/figures/eda/bbox_area_distribution.png   |  Bin 0 -> 41877 bytes
 reports/figures/eda/boxes_by_class_and_split.png |  Bin 0 -> 40245 bytes
 reports/figures/eda/image_type_distribution.png  |  Bin 0 -> 47407 bytes
 reports/figures/eda/samples_fire.png             |  Bin 0 -> 351134 bytes
 reports/figures/eda/samples_negative.png         |  Bin 0 -> 1827922 bytes
 reports/figures/eda/samples_smoke.png            |  Bin 0 -> 1613831 bytes
 reports/figures/eda/samples_smoke_and_fire.png   |  Bin 0 -> 1158368 bytes
 reports/results/eda/boxes_summary.csv            |    7 +
 reports/results/eda/dfire_colab_used.yaml        |    8 +
 reports/results/eda/image